# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. You'll learn to load the dataset, review its schema and record sets (with `@id` references), and extract and analyze the data using pandas.

### Dataset Source

The data is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant --quiet

## 1. Data Loading

Let's load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}\nLicense: {metadata.license}\n" if hasattr(metadata, 'version') and hasattr(metadata, 'license') else "")

## 2. Data Overview

Now, let's review the available record sets and their fields (using their `@id` values).

In [ ]:
# List all record sets available in the dataset, with their IDs and field information
from pprint import pprint

record_set_ids = []
print("Available Record Sets and their fields:\n")

for recordset in metadata.record_sets:
    print(f"- Record Set Name: {recordset.name}")
    print(f"  @id: {recordset.id}")
    record_set_ids.append(recordset.id)
    print(f"  Fields:")
    for field in recordset.fields:
        field_info = f"    - {field.name} (@id: {field.id}, dtype: {getattr(field, 'data_type', 'N/A')})"
        print(field_info)
    print()

## 3. Data Extraction

Let's load the records for one or more record sets into pandas DataFrames. All references (record sets, fields) are performed by their `@id` values, as shown above.

In [ ]:
dataframes = {}
print("Loading records for each record set...\n")

for rset_id in record_set_ids:
    print(f"Loading {rset_id} ...")
    records = list(dataset.records(record_set=rset_id))
    if len(records) > 0:
        dataframes[rset_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records, fields: {list(dataframes[rset_id].columns)}\n")
    else:
        print(f"No records found for {rset_id}\n")

# As an example, pick the first available record set for demonstration
if record_set_ids:
    main_record_set = record_set_ids[0]
    print(f"Columns for record set '{main_record_set}': {dataframes[main_record_set].columns.tolist()}\n")
    display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic EDA on a numeric field. We'll filter rows above a threshold, normalize the column, and group by another field if suitable fields are available.

In [ ]:
# Automatically choose a numeric column and a group-by column from the main dataframe
import numpy as np

df = dataframes[main_record_set]

# Detect numeric columns (float/int)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    # Try to infer numeric columns by attempting conversion
    possible_numeric = []
    for col in df.columns:
        try:
            df[col + "_num"] = pd.to_numeric(df[col], errors='coerce')
            if df[col + "_num"].notnull().sum() > 0:
                possible_numeric.append(col)
        except Exception:
            pass
    numeric_cols = possible_numeric
    if numeric_cols:
        numeric_field = numeric_cols[0]
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
else:
    numeric_field = numeric_cols[0]

print(f"Numeric field selected: '{numeric_field}'\n")

# Filtering for values above a threshold (e.g., 10)
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows\n")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a suitable categorical column
group_field = None
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
for col in categorical_cols:
    # Pick a groupable field with reasonably low number of unique values
    if df[col].nunique() > 1 and df[col].nunique() < 30:
        group_field = col
        break
if group_field:
    print(f"\nGrouping by '{group_field}':\n")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    display(grouped_df.head())
else:
    print("No suitable categorical column found for grouping.")

## 5. Visualization

Let's visualize the numeric field distribution and, if grouped, also show group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of '{numeric_field}' (filtered > {threshold})")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping was possible, plot group-wise means
if group_field:
    plt.figure(figsize=(10, 4))
    grouped = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    sns.barplot(x=grouped.index, y=grouped.values)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we've:

- Loaded and inspected the FAIR^2 dataset defined by a Croissant schema.
- Explored record sets, fields, and loaded data using their `@id`.
- Applied basic filtering and normalization, then grouped and visualized the data.

For further analysis, refer to the Croissant schema and documentation for detailed field descriptions and explore advanced data processing and ML applications.